### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.


In [4]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Why do parrots talk?")
response


AIMessage(content='### Short answer  \nParrots *talk* because they are natural **vocal learners**. Their brains and vocal anatomy allow them to copy sounds they hear, and they do this to communicate, bond, and get attention.  \n\n---\n\n## 1. What makes parrots “talk” so well?\n\n| Feature | How it helps parrots mimic |\n|---------|----------------------------|\n| **Syrinx** | The parrot’s “voice box” is located at the base of the trachea and can produce two independent sounds at once, giving them a wide range of tones. |\n| **Brain circuitry** | The avian vocal‑learning brain (especially the *nucleus interface* and *nucleus interfacialis*) is similar to the human speech‑planning areas. This lets parrots encode and replay sounds. |\n| **Social species** | Many parrot species live in flocks or pair‑bonded pairs. Mimicry helps them stay connected, establish dominance, or attract mates. |\n| **Curiosity & play** | Parrots are highly curious. Mimicking is a form of play that keeps them men

In [6]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in  {location}"

model_with_tools=model.bind_tools([get_weather])

In [7]:
response=model_with_tools.invoke("Whats the weather like in Boston?")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


Tool: get_weather
Args: {'location': 'Boston'}


### Tool ExeCUTION Loop

In [8]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."


It’s sunny in Boston.


In [9]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather function.', 'tool_calls': [{'id': 'fc_df7f222a-7523-468c-a750-4d6119cd238a', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 126, 'total_tokens': 158, 'completion_time': 0.034994584, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.006070651, 'prompt_tokens_details': None, 'queue_time': 0.471211451, 'total_time': 0.041065235}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_34cb00738d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1f38a-aaf1-73a0-8fcc-1ed141d425f8-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_df7f222a-7523-468c-a750-4d6119cd238a', 'type': 'tool_cal